In [1]:
# =====================================================================
# HITO 3: EXPANSIÓN, AUDITORÍA Y CARGA DEL MODELO EN PRODUCCIÓN
# =====================================================================

# 1. IMPORTACIÓN DE LIBRERÍAS DE HARDWARE Y MÉTRICAS
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("⏳ Inicializando el entorno de explotación independiente...")

# 2. CONFIGURACIÓN DE RUTAS FÍSICAS
raiz_proyecto = Path.cwd().parent
ruta_modelo = raiz_proyecto / "models" / "pipeline_xgboost.joblib"
ruta_dataset = raiz_proyecto / "datasets" / "importado" / "dataset_final.csv"

# =====================================================================
# FASE 1: CARGA DEL ARTEFACTO PIPELINE (.JOBLIB)
# =====================================================================
if ruta_modelo.exists():
    # Levantamos del congelador matemático el pipeline completo (Preprocesamiento + XGBoost)
    pipeline_xgb_cargado = joblib.load(ruta_modelo)
    print(f"✅ PIPELINE MAESTRO RECONSTRUIDO: {ruta_modelo.name}")
    print(f"   -> Componentes internos: {list(pipeline_xgb_cargado.named_steps.keys())}")
else:
    print(f"⚠️ Error crítico: No se encuentra el archivo binario en {ruta_modelo}. Asegúrate de haber ejecutado la sección 5.5 del Hito 2.")

# =====================================================================
# FASE 2: CARGA Y PREPARACIÓN DE LOS DATOS DE AUDITORÍA
# =====================================================================
if ruta_dataset.exists():
    df = pd.read_csv(ruta_dataset)
    
    # Separamos el target (y) de los predictores en bruto (X)
    y_real_log = df['price']
    X_bruto = df.drop(columns=['price'])
    
    # Nota: Para auditar exactamente el Test del 20%, replicamos la partición aleatoria fija
    from sklearn.model_selection import train_test_split
    _, X_test_bruto, _, y_test_log = train_test_split(
        X_bruto, y_real_log, 
        test_size=0.20, 
        random_state=42
    )
    print(f"✅ Set de validación ciega (20%) aislado con éxito: {X_test_bruto.shape[0]:,} registros.")
else:
    print(f"⚠️ Error crítico: Dataset no encontrado en {ruta_dataset}")

# =====================================================================
# FASE 3: EJECUCIÓN DE PREDICCIONES Y AUDITORÍA DE BONDADES
# =====================================================================
print("\n🔮 Lanzando inferencia masiva y evaluando bondad de ajuste...")
print("=" * 80)

# 1. El pipeline recibe los datos de texto y números EN BRUTO y los transforma sobre la marcha
predicciones_log = pipeline_xgb_cargado.predict(X_test_bruto)

# 2. Reversión estricta de la escala logarítmica analítica a Euros reales por noche
y_test_euros = np.expm1(y_test_log)
predicciones_euros = np.expm1(predicciones_log)

# 3. Cálculo de indicadores sobre la escala económica real de negocio
mae_test = mean_absolute_error(y_test_euros, predicciones_euros)
rmse_test = np.sqrt(mean_squared_error(y_test_euros, predicciones_euros))

# El R² se evalúa sobre la escala de varianza nativa en la que fue entrenado
r2_test = r2_score(y_test_log, predicciones_log)

# 4. DEPLOY DEL REPORTE FINAL POR CONSOLA
print(f"📊 REPORTE DE BONDAD DE AJUSTE DEL MODELO CARGADO (XGBOOST)")
print("-" * 80)
print(f"   -> MAE  (Desviación Media Real)  : {mae_test:.2f}€ por noche de error medio.")
print(f"   -> RMSE (Penalización Outliers) : {rmse_test:.2f}€ de varianza penalizada.")
print(f"   -> R²   (Coef. Determinación)   : {r2_test * 100:.2f}% de la varianza explicada.")
print("-" * 80)
if r2_test > 0.70:
    print("🏆 AUDITORÍA COMPLETADA: El archivo binario mantiene su alta precisión técnica (71%+).")
print("=" * 80)

⏳ Inicializando el entorno de explotación independiente...


KeyError: 118